# 1차 검색 방식 비교: ① 조 + 전결항목 1차 → 하위 항목 펼쳐 리랭크 vs ② 모든 조각 1차 → 리랭크 (Colab GPU)

임직원 골든셋(`RAG/eval/golden/golden_set.json`, dev·test)만 써요. 검색 코드·튜닝값은 11/19 B안 노트북(`hang_long_article_eval.ipynb`)과 같아요:
Dense(BGE-M3) + BM25(Kiwi, backend `rewrite`) → 가중 RRF(법령 ×0.6) → 후보 40 → bge-reranker-v2-m3. 내부규정 가점 0.

| 방식 | 1차 검색 대상 | 리랭커에 넣는 것 | LLM에 넘기는 5개 |
| --- | --- | --- | --- |
| **① 조+전결 → 펼치기** | 조문 단위 조각(조문·별표·서식안내·부칙, DB `data/regs`·`data/laws`) **+ 전결항목**(#25) | 후보 조를 **항으로 펼치고**, 연결된 **별표**(`appendix_links`)·**서식안내**(`form_links`), **인용된 조**(`article_refs` 따른다·준용, 예외는 반대 방향)를 더함. 별표 하나당 3조각·전체 200쌍까지(임베딩 유사도 순) | 조마다 항 1,200자 + 별표 2조각 + 서식안내 1 |
| **② 모든 조각 → 리랭크 (40)** | 항 단위 검색 조각 전부(6,330개: 항·별표·서식안내·부칙·전결항목) | 1차 상위 40조각 그대로 | 조(article_id)로 묶어 5개, 묶음마다 1,200자 |
| ② (200) | 같음 | 1차 상위 **200**조각 (①과 리랭커 예산을 맞춤) | 같음 |

- 1차에서 별표·서식안내가 잡히면 ①은 **연결된 조를 후보로 올리고**(별표는 그 조에 딸려 들어감), 연결이 없거나 전결항목·부칙이면 따로 후보가 돼요.
- 조 점수 = 그 조에 딸린 항·별표·서식 중 **가장 높은 리랭커 점수**. 인용으로 더해진 조는 1차 상위 10개 조에서만, 조당 3개까지, 한 단계만.

**지표**
- **정답 항 전달률**: `gold_hang`(정답 항 조각)의 앞 30자·끝 30자가 LLM에 넘기는 5개 글에 **모두** 들어 있는 비율 (항 정답이 확정된 문항만)
- **정답 조문 recall@5**: `gold_articles`(정답 조문·별표 id)가 LLM에 넘기는 5개에 들어간 비율 (별표가 조에 딸려 들어가도 맞음)
- **recall@20 / MRR**: 후보 순서에서 정답 조문 위치
- **LLM 입력 글자수**, **리랭커 쌍 수**, **질문당 검색 시간 p50·p95**(LLM 답변 생성 제외)
- ①에서 **인용·예외로만 들어온 조가 정답을 살린 문항 수**

런타임 → 런타임 유형 변경 → **GPU**(T4 이상) → 모두 실행. 코퍼스 임베딩 포함 T4 기준 약 20~30분.

In [ ]:
!pip install -q -U "sentence-transformers>=3.4" rank_bm25 kiwipiepy python-dotenv

In [ ]:
import os, sys, json, re, time, importlib.util, subprocess
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, torch

ROOT = Path(os.environ.get("LOCAL_ROOT", "/content/hr_chatbot"))   # LOCAL_ROOT: 내 컴퓨터에서 시험할 때만
ROOT.mkdir(parents=True, exist_ok=True)

CFG = {"embed_model": "BAAI/bge-m3", "rerank_model": "BAAI/bge-reranker-v2-m3",
       "k_dense": 30, "k_bm25": 30, "k_candidates": 40, "rrf_c": 60,
       "bm25_weight": 1.0, "dense_weight": 1.0, "law_weight": 0.6, "rewrite_bm25": True,
       "k_context": 5, "max_chars_per_article": 1200,
       # ① 펼치기 상한
       "ref_top_units": 10, "ref_per_unit": 3, "appx_chunks": 3, "pair_cap": 200,
       # LLM에 붙이는 하위 항목
       "ctx_appx": 2, "ctx_form": 1, "max_chars_appx": 1500,
       # ② 1차 후보 수
       "k_all_small": 40, "k_all_big": 200}

if "LOCAL_ROOT" not in os.environ:
    for repo in ("backend", "DB", "RAG"):
        d = ROOT / repo
        if d.exists():
            subprocess.run(f"rm -rf {d}", shell=True)
        subprocess.run(f"git clone -q --depth 1 https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git {d}", shell=True, check=True)
for repo in ("backend", "DB", "RAG"):
    print(f"{repo:8s}", subprocess.run(f"git -C {ROOT / repo} log -1 --format='%h %ad %s' --date=short", shell=True,
                                       capture_output=True, text=True).stdout.strip())
BACKEND, DB, HYERIM = ROOT / "backend", ROOT / "DB", ROOT / "RAG"

# 팀 채점 스크립트 (골든셋·dev/test 분할)
os.environ["BACKEND_DIR"] = str(BACKEND)
spec = importlib.util.spec_from_file_location("eval_golden", HYERIM / "scripts/eval_golden.py")
eg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eg)
ITEMS = eg.load_items()
SPLIT = eg.get_splits(ITEMS)

# backend 검색어 보강(glossary). DB 없이 불러옴
os.environ.update({"HRDB_URL": "", "SEARCH_MODE": "keyword", "LLM_BASE_URL": "", "PSEUDO_KEY": "colab-notebook-only"})
sys.path.insert(0, str(BACKEND))
from app.graph import retriever as be_retriever
print(f"골든셋 {len(ITEMS)}문항 · 재작성 예: {be_retriever.rewrite('결혼하면 경조휴가 며칠이야?')}")

In [ ]:
def jl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
def gold_of(it, key): return (it.get(key) or {}).get("primary") or []
def squash(s): return re.sub(r"\s+", "", s or "")
def dedupe(xs): return list(dict.fromkeys(xs))

def as_chunk(r):
    return {"id": r["chunk_id"], "doc_id": r["document_id"], "doc_name": r["doc_name"],
            "doc_kind": r.get("doc_kind") or ("법령" if r["chunk_id"].startswith("L") else "내부규정"),
            "type": r.get("chunk_type", ""), "article": (r.get("article") or "").strip(),
            "article_title": (r.get("article_title") or "").strip(), "text": r["text"],
            "article_id": r.get("article_id"),
            "embed_text": f"{r['doc_name']} {r.get('article') or ''} {r.get('article_title') or ''}".strip() + "\n" + r["text"]}

# 항 단위 검색 조각 전부 (②의 1차 대상, ①의 리랭크 대상)
NEWC = [as_chunk(r) for r in jl(DB / "data/hang/chunks.jsonl") if r.get("searchable")]
NEW_POS = {c["id"]: j for j, c in enumerate(NEWC)}
NEW_BY_ART = defaultdict(list)
for j, c in enumerate(NEWC):
    NEW_BY_ART[c["article_id"]].append(j)

# ①의 1차 대상: 조문 단위 조각(예전 그대로) + 전결항목
O2N = json.load(open(DB / "data/hang/old_to_new.json", encoding="utf-8"))
M1C = [as_chunk(r) for p in ("data/regs/chunks.jsonl", "data/laws/chunks.jsonl") for r in jl(DB / p) if r.get("searchable")]
for c in M1C:
    c["article_id"] = O2N[c["id"]]["article_id"]
M1C += [c for c in NEWC if c["type"] == "전결항목"]

ARTICLES = {a["article_id"]: a for a in jl(DB / "data/hang/articles.jsonl")}      # 조 (별표·부칙 없음)
AL = json.load(open(DB / "data/hang/appendix_links.json", encoding="utf-8"))
FL = json.load(open(DB / "data/hang/form_links.json", encoding="utf-8"))
APPX_OF, ART_OF_APPX = AL["article_to_appendix"], AL["appendix_to_article"]
FORM_OF, ART_OF_FORM = FL["article_to_form"], FL["form_to_article"]
REFS = json.load(open(DB / "data/hang/article_refs.json", encoding="utf-8"))["refs"]
REF_FWD, EXC_REV = defaultdict(list), defaultdict(list)
for r in REFS:
    if r["kind"] in ("따른다", "준용"):
        REF_FWD[r["from"]].append(r["to"])
    elif r["kind"] == "예외":
        EXC_REV[r["to"]].append(r["from"])    # 기본 조(to)가 뽑히면 예외 조(from)를 더함
REF_FWD = {k: dedupe(v) for k, v in REF_FWD.items()}
EXC_REV = {k: dedupe(v) for k, v in EXC_REV.items()}

print(f"① 1차 대상 {len(M1C)}조각 ({dict(Counter(c['type'] for c in M1C))})")
print(f"② 1차 대상 {len(NEWC)}조각 ({dict(Counter(c['type'] for c in NEWC))})")
print(f"조 {len(ARTICLES)} · 별표 연결 {sum(map(len, APPX_OF.values()))} · 서식 연결 {sum(map(len, FORM_OF.values()))} · "
      f"인용(따른다·준용) {sum(map(len, REF_FWD.values()))} · 예외 {sum(map(len, EXC_REV.values()))}")

# 골든셋 정답 id 가 지금 데이터에 있는지 (DB 가 바뀌면 어긋날 수 있음)
miss_art = Counter(a for it in ITEMS for a in gold_of(it, "gold_articles") if a not in NEW_BY_ART)
miss_hang = Counter(c for it in ITEMS for c in gold_of(it, "gold_hang") if c not in NEW_POS)
print("지금 데이터에 없는 정답 조문 id:", dict(miss_art) or "없음")
print("지금 데이터에 없는 정답 항 id:", dict(miss_hang) or "없음")

In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음 (느려요)")

def _load(cls, name, **kw):
    try:
        return cls(name, device=DEVICE, model_kwargs={"torch_dtype": torch.float16 if HAS_GPU else torch.float32}, **kw)
    except TypeError:
        m = cls(name, device=DEVICE, **kw)
        if HAS_GPU:
            m.model.half() if hasattr(m, "model") else m.half()
        return m

kiwi = Kiwi()
_KEEP = {"NNG", "NNP", "NR", "SL", "SN", "SH", "XR", "VV", "VA"}
def ko_tokens(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)
    return [t.form + ("다" if t.tag in ("VV", "VA") else "") for t in kiwi.tokenize(text) if t.tag in _KEEP]

EMBEDDER = _load(SentenceTransformer, CFG["embed_model"])
EMBEDDER.max_seq_length = 1024
RERANKER = _load(CrossEncoder, CFG["rerank_model"], max_length=1024)

# 같은 글은 한 번만 임베딩 (① 조문 조각과 ② 항 조각이 겹치는 경우가 많음)
_TCACHE = {}
def embed_texts(texts):
    todo = [t for t in dict.fromkeys(texts) if t not in _TCACHE]
    if todo:
        for t, v in zip(todo, EMBEDDER.encode(todo, batch_size=32, normalize_embeddings=True,
                                               convert_to_numpy=True, show_progress_bar=True)):
            _TCACHE[t] = v.astype(np.float32)
    return np.stack([_TCACHE[t] for t in texts])

IDX = {}
for name, chunks in (("m1", M1C), ("all", NEWC)):
    t0 = time.time()
    IDX[name] = {"C": chunks, "BM25": BM25Okapi([ko_tokens(c["embed_text"]) for c in chunks]),
                 "EMB": embed_texts([c["embed_text"] for c in chunks])}
    print(f"{name}: {len(chunks)}조각 BM25 + 임베딩 {time.time() - t0:.0f}초")
NEW_EMB = IDX["all"]["EMB"]

In [ ]:
# ── ② 검색 · ③ 리랭커 (11/19 노트북과 같은 계산) ──
_QCACHE, _RCACHE = {}, {}
RR = {"pairs": 0, "sec": 0.0}

def embed_query(q):
    if q not in _QCACHE:
        _QCACHE[q] = EMBEDDER.encode([re.sub(r"\[[^\]]*\]", " ", q)], normalize_embeddings=True,
                                     convert_to_numpy=True)[0].astype(np.float32)
    return _QCACHE[q]

def candidates(name, question, n):
    ix = IDX[name]
    bq = be_retriever.rewrite(question) if CFG["rewrite_bm25"] else question
    bs = ix["BM25"].get_scores(ko_tokens(bq))
    bm = [int(i) for i in np.argsort(-bs)[:max(CFG["k_bm25"], n)] if bs[i] > 0]
    de = [int(i) for i in np.argsort(-(ix["EMB"] @ embed_query(question)))[:max(CFG["k_dense"], n)]]
    score = defaultdict(float)
    for lst, w in ((bm, CFG["bm25_weight"]), (de, CFG["dense_weight"])):
        for r, i in enumerate(lst):
            score[i] += w / (CFG["rrf_c"] + r + 1)
    for i in score:
        if ix["C"][i]["doc_kind"] == "법령":
            score[i] *= CFG["law_weight"]
    return sorted(score, key=score.get, reverse=True)[:n]

def rerank(question, js):
    # js: NEWC 번호들 → {번호: 점수 0~1}
    q = re.sub(r"\[[^\]]*\]", " ", question)
    todo = [j for j in dict.fromkeys(js) if (q, j) not in _RCACHE]
    if todo:
        t0 = time.perf_counter()
        s = np.asarray(RERANKER.predict([(q, NEWC[j]["embed_text"]) for j in todo], batch_size=16,
                                        convert_to_numpy=True, show_progress_bar=False), dtype=float).reshape(-1)
        if s.min() < 0 or s.max() > 1:
            s = 1 / (1 + np.exp(-s))
        _RCACHE.update({(q, j): float(v) for j, v in zip(todo, s)})
        RR["pairs"] += len(todo)
        RR["sec"] += time.perf_counter() - t0
    return {j: _RCACHE[(q, j)] for j in js}

def sims(question, js):
    v = embed_query(question)
    return {j: float(NEW_EMB[j] @ v) for j in js}

## ① 조 + 전결항목 1차 → 하위 항목 펼쳐 리랭크
```
1차: 조문 단위 조각 + 전결항목 → 후보 40조각 → 후보 단위로 묶음
     조문 조각 → 그 조 / 별표·서식안내 → 연결된 조(없으면 그대로) / 전결항목·부칙 → 그대로
인용: 상위 10개 조의 '따른다·준용' 대상 조(조당 3개, 임베딩 유사도 순) + '예외' 조(반대 방향) 추가, 한 단계만
펼치기: 조 → 항 + 연결 별표(별표당 3조각) + 연결 서식안내 / 따로 후보 → 그 조각들(별표면 3조각)
상한: 후보마다 가장 비슷한 조각 하나는 꼭 넣고, 나머지는 유사도 순으로 전체 200쌍까지
리랭크 1번 → 후보 점수 = 딸린 조각 중 최고점 → 순서
```

In [ ]:
def is_article(a): return a in ARTICLES

def units_m1(question):
    C1 = IDX["m1"]["C"]
    units, via = [], {}
    def add(u, how):
        if u not in NEW_BY_ART:
            return
        if u not in via:
            units.append(u); via[u] = how
        elif how not in via[u]:
            via[u] += "+" + how
    for i in candidates("m1", question, CFG["k_candidates"]):
        a = C1[i]["article_id"]
        if is_article(a):
            add(a, "1차")
        elif a in ART_OF_APPX:                      # 별표 → 연결된 조 (별표는 그 조에 딸려 들어감)
            for t in ART_OF_APPX[a][:2]:
                add(t, "1차(별표)")
        elif a in ART_OF_FORM:
            for t in ART_OF_FORM[a][:2]:
                add(t, "1차(서식)")
        else:
            add(a, "1차")                           # 전결항목·부칙·연결 없는 별표·서식
    # 인용된 조 (상위 10개 조에서만, 한 단계)
    top = [u for u in units if is_article(u)][:CFG["ref_top_units"]]
    for u in top:
        for kind, cands in (("인용", REF_FWD.get(u, [])), ("예외", EXC_REV.get(u, []))):
            cands = [t for t in cands if t in NEW_BY_ART]
            if not cands:
                continue
            best = sorted(cands, key=lambda t: -max(sims(question, NEW_BY_ART[t]).values()))
            for t in best[:CFG["ref_per_unit"]]:
                add(t, kind)
    return units, via

def expand(question, u):
    # 후보 하나 → (본문 조각, 딸린 조각) NEWC 번호
    body = list(NEW_BY_ART[u])
    attach = []
    if is_article(u):
        for x in APPX_OF.get(u, []):
            js = NEW_BY_ART.get(x, [])
            s = sims(question, js)
            attach += sorted(js, key=lambda j: -s[j])[:CFG["appx_chunks"]]
        for x in FORM_OF.get(u, []):
            attach += [j for j in NEW_BY_ART.get(x, []) if NEWC[j]["type"] == "서식안내"]
    elif NEWC[body[0]]["type"] == "별표":
        s = sims(question, body)
        body = sorted(body, key=lambda j: -s[j])[:CFG["appx_chunks"]]
    return body, attach

def search_m1(question):
    units, via = units_m1(question)
    parts = {u: expand(question, u) for u in units}
    every = dedupe(j for u in units for j in parts[u][0] + parts[u][1])
    s = sims(question, every)
    keep = set()
    for u in units:                                  # 후보마다 가장 비슷한 조각 하나는 꼭
        js = parts[u][0] + parts[u][1]
        keep.add(max(js, key=lambda j: s[j]))
    for j in sorted(every, key=lambda j: -s[j]):
        if len(keep) >= CFG["pair_cap"]:
            break
        keep.add(j)
    sc = rerank(question, [j for j in every if j in keep])
    ranked = []
    for u in units:
        b = [j for j in parts[u][0] if j in sc]
        a = [j for j in parts[u][1] if j in sc]
        if b or a:
            ranked.append({"id": u, "score": max(sc[j] for j in b + a), "body": b, "attach": a, "via": via[u]})
    ranked.sort(key=lambda r: -r["score"])
    return ranked, sc

## ② 모든 조각 1차 → 리랭크
```
1차: 항 단위 검색 조각 전부(항·별표·서식안내·부칙·전결항목) → 후보 K조각 (K = 40 / 200)
리랭크 → 조(article_id)로 묶음, 묶음 점수 = 최고점 → 순서. 펼치기·연결표 없음
```

In [ ]:
def search_m2(question, k):
    js = candidates("all", question, k)
    sc = rerank(question, js)
    groups = defaultdict(list)
    for j in js:
        groups[NEWC[j]["article_id"]].append(j)
    ranked = [{"id": a, "score": max(sc[j] for j in g), "body": g, "attach": [], "via": "1차"} for a, g in groups.items()]
    ranked.sort(key=lambda r: -r["score"])
    return ranked, sc

def head_of(c): return f"{c['article']}({c['article_title']})" if c["article_title"] else c["article"]

def context(ranked, sc, k=None):
    # 두 방식 같은 규칙: 본문은 점수 높은 조각부터 이어 붙여 1,200자, 딸린 별표 2조각·서식안내 1개
    out, seen = [], set()                            # seen: 같은 별표가 조 두 곳에 딸려도 한 번만 넣음
    for r in ranked[:k or CFG["k_context"]]:
        js = sorted(r["body"], key=lambda j: -sc[j])
        texts = []
        for n_, j in enumerate(js):
            t, h = NEWC[j]["text"], head_of(NEWC[j])
            texts.append(t[len(h):].lstrip() if n_ and h and t.startswith(h) else t)
        text = "\n".join(texts)
        cap = CFG["max_chars_per_article"]
        text = text if len(text) <= cap else text[:cap] + " …(생략)"
        covered = {r["id"]}
        att = sorted(r["attach"], key=lambda j: -sc[j])
        appx = [j for j in att if NEWC[j]["type"] != "서식안내" and j not in seen][:CFG["ctx_appx"]]
        form = [j for j in att if NEWC[j]["type"] == "서식안내" and j not in seen][:CFG["ctx_form"]]
        seen.update(appx + form)
        for j in appx + form:
            t = NEWC[j]["text"]
            text += f"\n[{NEWC[j]['article']}]\n" + (t if len(t) <= CFG["max_chars_appx"] else t[:CFG["max_chars_appx"]] + " …(생략)")
            covered.add(NEWC[j]["article_id"])
        out.append({"id": r["id"], "text": text, "covered": covered, "via": r["via"], "score": round(r["score"], 4)})
    return out

q = "며느리가 출산하면 할아버지인 저도 휴가 있어요?"
for label, fn in (("①", search_m1), ("② 40", lambda x: search_m2(x, CFG["k_all_small"]))):
    rk, sc = fn(q)
    print(label)
    for h in context(rk, sc, 3):
        print(f"  {h['score']:.3f} {h['id']} ({h['via']}) · {len(h['text'])}자 · {squash(h['text'])[:70]}")

In [ ]:
def hang_sig(cid):
    c = NEWC[NEW_POS[cid]]
    body, h = c["text"], head_of(c)
    s = squash(body[len(h):] if h and body.startswith(h) else body)
    return (s[:30], s[-30:]) if len(s) > 30 else (s, s)

def delivered(cid, blob):
    if cid not in NEW_POS:
        return False
    a, b = hang_sig(cid)
    return a in blob and b in blob

METHODS = {"① 조+전결 → 펼치기": search_m1,
           "② 모든 조각 (40)": lambda q: search_m2(q, CFG["k_all_small"]),
           "② 모든 조각 (200)": lambda q: search_m2(q, CFG["k_all_big"])}
SPLITS = ["dev", "test"]
ROWS = []
for split in SPLITS:
    QS = [it for it in ITEMS if SPLIT[it["id"]] == split and gold_of(it, "gold_articles")]
    for label, fn in METHODS.items():
        fn("연차는 며칠이야?")                          # 예열 (GPU 첫 호출은 빼고 잼)
        t0 = time.time()
        for it in QS:
            _QCACHE.clear(); _RCACHE.clear(); RR.update(pairs=0, sec=0.0)
            if HAS_GPU: torch.cuda.synchronize()
            tq = time.perf_counter()
            ranked, sc = fn(it["question"])
            ctx = context(ranked, sc)
            if HAS_GPU: torch.cuda.synchronize()
            ms = (time.perf_counter() - tq) * 1000
            blob = squash("\n".join(h["text"] for h in ctx))
            gold = gold_of(it, "gold_articles")
            cov5 = set().union(*[h["covered"] for h in ctx]) if ctx else set()
            order = [{r["id"]} | {NEWC[j]["article_id"] for j in r["attach"]} for r in ranked]
            first = next((n + 1 for n, s_ in enumerate(order) if s_ & set(gold)), None)
            gh = it.get("gold_hang") or {}
            hang_ok = gh.get("auto") in (True, "checked") and bool(gh.get("primary"))
            hit_ctx = [h for h in ctx if h["covered"] & set(gold)]
            ROWS.append({"split": split, "방식": label, "id": it["id"], "question": it["question"],
                         "정답 항 전달률": np.mean([delivered(c, blob) for c in gh["primary"]]) if hang_ok else np.nan,
                         "정답 조문 recall@5": len(set(gold) & cov5) / len(gold),
                         "정답 조문 recall@20": len(set(gold) & set().union(*order[:20])) / len(gold) if order else 0.0,
                         "MRR": 1 / first if first else 0.0,
                         "인용으로만 살림": bool(hit_ctx) and all(h["via"] in ("인용", "예외") for h in hit_ctx),
                         "LLM 입력 글자수": sum(len(h["text"]) for h in ctx),
                         "ms": ms, "리랭커 쌍 수": RR["pairs"], "리랭커 ms": RR["sec"] * 1000,
                         "ctx": [f"{h['id']}({h['via']})" for h in ctx], "gold": gold})
        print(f"{split} {label}: {len(QS)}문항 {time.time() - t0:.0f}초")
R = pd.DataFrame(ROWS)

## 결과

In [ ]:
KEYS = ["정답 항 전달률", "정답 조문 recall@5", "정답 조문 recall@20", "MRR", "LLM 입력 글자수", "리랭커 쌍 수"]
def summary(df):
    g = df.groupby("방식", sort=False)
    s = g[KEYS].mean().round(3).reindex(list(METHODS))
    s["인용으로만 살린 문항"] = g["인용으로만 살림"].sum()
    s["문항"] = g.size()
    s["항 전달률 문항"] = g["정답 항 전달률"].count()
    return s

def speed(df):
    g = df.groupby("방식", sort=False)
    return pd.DataFrame({"p50 ms": g["ms"].median(), "p95 ms": g["ms"].quantile(0.95),
                         "리랭커 평균 ms": g["리랭커 ms"].mean(), "리랭커 쌍 평균": g["리랭커 쌍 수"].mean(),
                         "리랭커 쌍 최대": g["리랭커 쌍 수"].max()}).reindex(list(METHODS)).round(1)

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음")
for split in SPLITS + ["dev+test"]:
    D = R if split == "dev+test" else R[R["split"] == split]
    print(f"\n========== {split} ==========")
    display(summary(D))
    display(speed(D))

## 문항별로 갈린 것 (정답 항 전달률·정답 조문 recall@5 기준)

In [ ]:
def diff(a, b, key):
    A = R[R["방식"] == a].set_index("id"); B = R[R["방식"] == b].set_index("id")
    d = (A[key] - B[key]).dropna()
    win, lose = d[d > 0].index, d[d < 0].index
    print(f"■ {key}: {a} 이 나음 {len(win)}문항 · {b} 이 나음 {len(lose)}문항")
    for ids, who in ((win, a), (lose, b)):
        for i in list(ids)[:12]:
            print(f"  [{who[:1]}] {i} {A.loc[i, 'question'][:40]}")
            print(f"      정답 {A.loc[i, 'gold']}")
            print(f"      ① {A.loc[i, 'ctx']}")
            print(f"      ② {B.loc[i, 'ctx']}")

for key in ("정답 항 전달률", "정답 조문 recall@5"):
    diff("① 조+전결 → 펼치기", "② 모든 조각 (200)", key)
    print()

print("■ ①에서 인용·예외로만 정답이 들어온 문항")
display(R[(R["방식"] == "① 조+전결 → 펼치기") & R["인용으로만 살림"]][["split", "id", "question", "gold", "ctx"]])

In [ ]:
# 결과 파일 (원하면 내려받기)
out = Path("/content/first_stage_compare.csv") if "LOCAL_ROOT" not in os.environ else ROOT / "first_stage_compare.csv"
R.to_csv(out, index=False, encoding="utf-8-sig")
print("저장:", out)